## Import Requisite Lbraries

In [ ]:
import pandas as pd
import numpy as np
import sys
import os  # import operating system for dir
from ucimlrepo import fetch_ucirepo
python_version = f"Python {sys.version.split()[0]}"
python_version

## Ensure Directory

In [ ]:
from eda_toolkit import ensure_directory

base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data")
data_output = os.path.join(os.pardir, "data_output")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(data_output)
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

## UCI ML Repository

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

# Combine X and y into entire df
df = X.join(y, how="inner")

## Add Ids

In [ ]:
from eda_toolkit import add_ids

# Add a column of unique IDs with 9 digits and call it "census_id"
df = add_ids(
    df=df,
    id_colname="census_id",
    num_digits=9,
    seed=111,
    set_as_index=True,
)

df.head()

In [ ]:
if df.index.is_unique:
    print("The index is unique.")
else:
    print("The index is not unique.")

## Save Out the DataFrame

In [ ]:
df.to_csv(os.path.join(data_path, "adult_income.csv"))
df.to_parquet(os.path.join(data_path, "adult_income.parquet"))

## Read CSV w/ Progress Bar

### 1. `read_csv_with_progress` - basic

In [ ]:
print("\nReading CSV with progress bar...")

from eda_toolkit import read_csv_with_progress
df = read_csv_with_progress(os.path.join(data_path, "adult_income.csv"), low_memory=False).set_index("census_id")


### 2. `read_csv_with_progress` - nrows

In [ ]:
print("\n[2] read_csv_with_progress — nrows=500")
df_small = read_csv_with_progress(
    os.path.join(data_path, "adult_income.csv"),
    nrows=500,
).set_index("census_id")
df_small.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
assert len(df_small) == 500, f"Expected 500 rows, got {len(df_small)}"
print(f"  Shape: {df_small.shape}")

### 3. `read_csv_with_progress`: custom chunksize

In [ ]:
print("\n[3] read_csv_with_progress — chunksize=5000")
df_chunked = read_csv_with_progress(
    os.path.join(data_path, "adult_income.csv"),
    chunksize=5000,
).set_index("census_id")
df_chunked.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
print(f"  Shape: {df_chunked.shape}")

### 4. `read_csv_with_progress` - usecols kwarg

In [ ]:
print("\n[4] read_csv_with_progress — usecols kwarg")
df_cols = read_csv_with_progress(
    os.path.join(data_path, "adult_income.csv"),
    usecols=["age", "education", "income"],
)
print(f"  Columns: {list(df_cols.columns)}")
print(f"  Shape: {df_cols.shape}")

### 5. `read_csv_with_progress` - dtype kwarg

In [ ]:
print("\n[5] read_csv_with_progress — dtype kwarg")
df_typed = read_csv_with_progress(
    os.path.join(data_path, "adult_income.csv"),
    dtype={"age": float, "hours-per-week": float},
)
df_typed.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
print(f"  age dtype: {df_typed['age'].dtype}")
print(f"  hours-per-week dtype: {df_typed['hours-per-week'].dtype}")

## Detecting Outliers

In [ ]:
numeric_features = ["age", "fnlwgt", "education-num",
                    "capital-gain", "capital-loss", "hours-per-week"]

### 1. Basic IQR: all numeric features

In [ ]:
from eda_toolkit import detect_outliers

print("\n[1] IQR — all numeric features (threshold=1.5)")
summary = detect_outliers(df, method="iqr", threshold=1.5)
print(summary.to_string(index=False))

### 2. IQR: strict threshold

In [ ]:
print("\n[2] IQR — strict threshold (threshold=3.0)")
summary_strict = detect_outliers(
    df,
    features=numeric_features,
    method="iqr",
    threshold=3.0,
)
print(summary_strict.to_string(index=False))

### 3. Z-Score Method

In [ ]:
print("\n[3] Z-score — threshold=3.0")
summary_z = detect_outliers(
    df,
    features=numeric_features,
    method="zscore",
    threshold=3.0,
)
print(summary_z.to_string(index=False))

### 4. Isolation Forest

In [ ]:
print("\n[4] Isolation Forest — contamination=0.05")
summary_iso = detect_outliers(
    df,
    features=numeric_features,
    method="isoforest",
    contamination=0.05,
)
print(summary_iso.to_string(index=False))

### 5. `return_mask=True`

In [ ]:
print("\n[5] return_mask=True")
summary_mask, mask = detect_outliers(
    df,
    features=numeric_features,
    method="iqr",
    return_mask=True,
)
print(f"  Mask shape: {mask.shape}")
print(f"  Outlier rows per feature:\n{mask.sum()}")

### 6. `flag_col`

In [ ]:
print("\n[6] flag_col='is_outlier'")
df_flagged = df.copy()
detect_outliers(
    df_flagged,
    features=numeric_features,
    method="iqr",
    flag_col="is_outlier",
)
n_flagged = df_flagged["is_outlier"].sum()
pct_flagged = n_flagged / len(df_flagged) * 100

print(f"Total flagged rows: {n_flagged:,} ({pct_flagged:.1f}%)\n")
df_flagged[df_flagged["is_outlier"]].head(5)

### 7. groupby: within education groups

In [ ]:
print("\n[7] groupby='education' — IQR within each education level")
summary_grp = detect_outliers(
    df,
    features=["age", "hours-per-week", "capital-gain"],
    method="iqr",
    groupby="education",
)
print(summary_grp.to_string(index=False))

### 8. `groupby` + `return_mask` + `flag_col`

In [ ]:
print("\n[8] groupby + return_mask + flag_col combined")
df_grp = df.copy()
summary_grp2, mask_grp = detect_outliers(
    df_grp,
    features=["age", "hours-per-week"],
    method="iqr",
    groupby="education",
    return_mask=True,
    flag_col="is_outlier_grouped",
)
print(summary_grp2.to_string(index=False))
print(f"\n  Rows flagged (grouped): {df_grp['is_outlier_grouped'].sum():,}")

### 9. Feature-level filtering using mask

In [ ]:
print("\n[9] Feature-level filtering using mask")
summary_m, mask_m = detect_outliers(
    df,
    features=numeric_features,
    method="iqr",
    return_mask=True,
)
# Rows where capital-gain is specifically an outlier
capital_gain_outliers = df[mask_m["capital-gain"]]
print(f"  capital-gain outlier rows: {len(capital_gain_outliers):,}")
print(capital_gain_outliers[["age", "capital-gain", "income"]].head(10).to_string())

### 10. Co-occurring outliers: rows flagged in multiple features

In [ ]:
print("\n[10] Co-occurring outliers — rows flagged in multiple features")
summary_co, mask_co = detect_outliers(
    df,
    features=numeric_features,
    method="iqr",
    return_mask=True,
)
outlier_counts = mask_co.sum(axis=1)
print(outlier_counts.value_counts().sort_index().to_string())

print("\nAll detect_outliers usage examples completed successfully.")

### 11. `verbose=True`: ASCII summary report

In [ ]:
print("\n[11] verbose=True — ASCII summary report")
detect_outliers(
    df,
    features=numeric_features,
    method="iqr",
    threshold=1.5,
    verbose=True,
)

### 12. `return_bounds=True`: pipe bounds into `data_doctor`

In [ ]:
print("\n[12] return_bounds=True — bounds dict for downstream data_doctor use")
summary_b, bounds = detect_outliers(
    df,
    features=numeric_features,
    method="iqr",
    return_bounds=True,
)
print("  Bounds per feature:")
for feat, (lower, upper) in bounds.items():
    print(f"    {feat:<20} lower={lower:>10}  upper={upper:>10}")
 
print("\n  Example: pass capital-gain bounds directly to data_doctor:")
print(f"    lower_cutoff={bounds['capital-gain'][0]}")
print(f"    upper_cutoff={bounds['capital-gain'][1]}")

## Binning Numerical Columns

In [ ]:
bin_ages = [
    0,
    18,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    float("inf"),
]

In [ ]:
label_ages = [
    "< 18",
    "18-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70-79",
    "80-89",
    "90-99",
    "100 +",
]

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=bin_ages,
    labels=label_ages,
    right=False,
)

## Consolidate Income (Outcome) Column

In [ ]:
def outcome_merge(val):
    if val == "<=50K" or val == "<=50K.":
        return "<=50K"
    else:
        return ">50K"

In [ ]:
df["income"] = df["income"].apply(outcome_merge)

In [ ]:
df["income"].value_counts()

## Generate Table 1

In [ ]:
from eda_toolkit import generate_table1

table1 = generate_table1(
    df=df,
    value_counts=True,
    # max_categories=3,
    export_markdown=True,
    markdown_path=os.path.join(data_output, "table1_summary.md"),
)

# table1 = table1.drop(columns=["Type", "Mode"])

In [ ]:
table1

In [ ]:
print(table1)

In [ ]:
from eda_toolkit import generate_table1

# Get DataFrame and Markdown string
df1 = generate_table1(
    df,
    value_counts=True,
    decimal_places=0,
)

In [ ]:
# Just get markdown string (no tuple)
md_only = generate_table1(
    df,
    export_markdown=True,
    return_markdown_only=True,
    combine=False,
    include_types="continuous",
    markdown_path=os.path.join(data_output, "table1_summary.md"),
)

In [ ]:
table1 = generate_table1(
    df,
    export_markdown=True,
    combine=False,
    include_types="categorical",
    markdown_path=os.path.join(data_output, "table1_summary.md"),
)

In [ ]:
# Save Markdown to file
generate_table1(
    df,
    value_counts=True,
    export_markdown=True,
    markdown_path=os.path.join(data_output, "custom_prefix_table1_summary.md"),
    combine=False,
)

In [ ]:
# Get DataFrame and Markdown string
df2 = generate_table1(
    df,
    value_counts=True,
    include_types="categorical",
    # decimal_places=0,
    combine=True,
)

In [ ]:
df2

In [ ]:
print(df2)

In [ ]:
df3, df4 = generate_table1(
    df,
    value_counts=True,
    include_types="both",
    max_categories=3,
    export_markdown=True,
    combine=False,
    markdown_path=os.path.join(data_output, "both_table1_summary.md"),
)

df3 = df3.drop(columns=["Type", "Mode"])

In [ ]:
df3

In [ ]:
print(df3)

In [ ]:
df4

In [ ]:
print(df4)

In [ ]:
df4

### Table 1 with P-Values

In [ ]:
# 1) Equal-width bins

df["fnlwgt_bin"] = pd.cut(
    df["fnlwgt"], bins=5, labels=[f"Bin {i}" for i in range(1, 6)], include_lowest=True
)

# 2) Equal-frequency (quantile) bins
df["fnlwgt_bin_quantile"] = pd.qcut(
    df["fnlwgt"],
    q=5,
    labels=[f"Q{i}" for i in range(1, 6)],
    duplicates="drop",  # in case of ties
)

In [ ]:
df_table1_cat = df[["fnlwgt_bin", "age_group", "income"]]

In [ ]:
p_value_table_1_cat = generate_table1(
    df_table1_cat,
    value_counts=True,
    include_types="categorical",
    export_markdown=True,
    groupby_col="income",
    drop_columns=["Missing (n)", "Missing (%)", "income", "Type", "Mode"],
    drop_variables="income",
    markdown_path=os.path.join(data_output, "p_val_table1_summary.md"),
)

In [ ]:
p_value_table_1_cat

In [ ]:
print(p_value_table_1_cat)

## Trailing Period Removal

In [ ]:
from eda_toolkit import strip_trailing_period

# Create a sample dataframe with trailing periods in some values
data = {
    "values": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
}
df_trail = pd.DataFrame(data)

# Remove trailing periods from the 'values' column
df_trail = strip_trailing_period(df=df_trail, column_name="values")
df_trail

## Standardized Dates

In [ ]:
from eda_toolkit import parse_date_with_rule

# Sample date strings
date_strings = ["15/04/2021", "04/15/2021", "01/12/2020", "12/01/2020"]

# Standardize the date strings
standardized_dates = [parse_date_with_rule(date) for date in date_strings]

print(standardized_dates)

In [ ]:
# Creating the DataFrame
data = {
    "date_column": [
        "31/12/2021",
        "01/01/2022",
        "12/31/2021",
        "13/02/2022",
        "07/04/2022",
    ],
    "name": ["Alice", "Bob", "Charlie", "David", "Eve"],
    "amount": [100.0, 150.5, 200.75, 250.25, 300.0],
}

df_fake = pd.DataFrame(data)

# Apply the function to the DataFrame column
df_fake["standardized_date"] = df_fake["date_column"].apply(parse_date_with_rule)

print(df_fake)

## DataFrame Analysis

In [ ]:
from eda_toolkit import dataframe_profiler

dataframe_profiler(df=df, background_color="brown")

## Generating Summary Tables for Variable Combinations

In [ ]:
from eda_toolkit import summarize_all_combinations

# Define unique variables for the analysis
unique_vars = [
    "age_group",
    "workclass",
    "education",
    "occupation",
    "race",
    "sex",
    "income",
]

# Generate summary tables for all combinations of the specified variables
summary_tables, all_combinations = summarize_all_combinations(
    df=df,
    # data_path=data_output,
    variables=unique_vars,
    # data_name="census_summary_tables.xlsx",
)

print(all_combinations)

## Saving DataFrames to Excel with Customized Formatting

In [ ]:
from eda_toolkit import save_dataframes_to_excel

# Example usage
file_name = "df_census.xlsx"  # Name of the output Excel file
file_path = os.path.join(data_path, file_name)

# filter DataFrame to Ages 18-40
filtered_df = df[(df["age"] > 18) & (df["age"] < 40)]

df_dict = {
    "original_df": df,
    "ages_18_to_40": filtered_df,
}

save_dataframes_to_excel(
    file_path=file_path,
    df_dict=df_dict,
    decimal_places=0,
)

## Creating Contingency Tables

In [ ]:
from eda_toolkit import contingency_table

# Example usage
contingency_table(
    df=df,
    cols=[
        "age_group",
        "workclass",
        "race",
        "sex",
    ],
    sort_by=1,
)

## Highlighting Specific Columns in a DataFrame

In [ ]:
from eda_toolkit import highlight_columns

# Applying the highlight function
highlighted_df = highlight_columns(
    df=df.head(),
    columns=["age", "education"],
    color="brown",
)

highlighted_df

## Group Imputer Examples

In [ ]:
################################################################################
## UCI ML Repository
################################################################################

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

# Combine X and y into entire df
adult_df = X.join(y, how="inner")

print("Adult Income Dataset")
adult_df.head()

In [ ]:
################################################################################
## Introduce Missingness to 'age' column for demonstration
################################################################################

adult_df.loc[adult_df.sample(frac=0.3, random_state=42).index, "age"] = np.nan
print(f"\nProportion of missing values in 'age': {adult_df['age'].isna().mean()}\n")

### Example 1: GroupBy Imputer with Mean Statistic and Fixed Fallback

In [ ]:
from eda_toolkit import groupby_imputer

################################################################################
## 1. Impute with fallback='global'
################################################################################


X_global = groupby_imputer(
    df=adult_df,
    impute_col="age",
    by=["workclass", "education"],
    stat="mean",
    fallback="global",
    as_new_col=True,
)

print("\n### Head with global fallback ###")
print(X_global[["age", "age_mean_imputed"]].head())

print("Means:")
print(f"Original adult_df['age'].mean(): {adult_df['age'].mean()}")
print(f"Imputed (global fallback) mean: {X_global['age_mean_imputed'].mean()}")

In [ ]:
X_global.head()

### Example 2: Impute with fallback equal to a fixed number instead of global

In [ ]:
################################################################################
## 2. Impute with fallback equal to a fixed number instead of global
################################################################################

X_fixed = groupby_imputer(
    df=adult_df,
    impute_col="age",
    by=["workclass", "education"],
    stat="mean",
    fallback=50,  # <-- fixed fallback value
    as_new_col=True,
)

print("\n### Head with fixed fallback=50 ###")
print(X_fixed[["age", "age_mean_imputed"]].head())

print("Means:")
print(f"Original adult_df['age'].mean(): {adult_df['age'].mean()}")
print(f"Imputed (fixed fallback=50) mean: {X_fixed['age_mean_imputed'].mean()}")

In [ ]:
X_fixed.head()

### Wrap up comparison

In [ ]:
################################################################################
## Wrap up comparison
################################################################################

print("\n### Comparison summary ###")
print(f"Missingness in original age: {adult_df['age'].isna().mean()}")
print(f"Global fallback mean: {X_global['age_mean_imputed'].mean()}")
print(f"Fixed fallback=50 mean: {X_fixed['age_mean_imputed'].mean()}")

## Normality Tests

In [ ]:
from eda_toolkit import normality_tests

# All three tests, all numeric features
summary = normality_tests(df, decimal_places=4)

print(summary)

In [ ]:
# Specific features and tests
summary = normality_tests(
    df,
    features=["age", "capital-gain", "hours-per-week"],
    tests=["shapiro", "dagostino"],
    alpha=0.01,
    decimal_places=10
)

print(summary)

In [ ]:
summary

In [ ]:
summary = normality_tests(
    df_small,
    features=["bmi", "age", "blood_pressure"],
    tests=["shapiro", "anderson"],
    alpha=0.05,
)
print(summary)